In [2]:
import pandas as pd
from pathlib import Path

# If this notebook lives in notebooks/, the data folder is one level up
RAW = Path("../data/raw")
print(list(RAW.glob("*.csv")))

[WindowsPath('../data/raw/Most-Recent-Cohorts-Field-of-Study.csv'), WindowsPath('../data/raw/Most-Recent-Cohorts-Institution.csv')]


In [4]:
fos = pd.read_csv(RAW / "Most-Recent-Cohorts-Field-of-Study.csv",
                  dtype=str, low_memory=False)
print(fos.shape)

print([c for c in fos.columns if c.startswith(("EARN", "DEBT"))])
print([c for c in fos.columns if c.startswith(("CIP", "CRED", "UNITID", "INSTNM", "CONTROL"))])

(223587, 230)
['DEBT_ALL_STGP_ANY_N', 'DEBT_ALL_STGP_ANY_MEAN', 'DEBT_ALL_STGP_ANY_MDN', 'DEBT_ALL_STGP_EVAL_N', 'DEBT_ALL_STGP_EVAL_MEAN', 'DEBT_ALL_STGP_EVAL_MDN', 'DEBT_ALL_PP_ANY_N', 'DEBT_ALL_PP_ANY_MEAN', 'DEBT_ALL_PP_ANY_MDN', 'DEBT_ALL_PP_EVAL_N', 'DEBT_ALL_PP_EVAL_MEAN', 'DEBT_ALL_PP_EVAL_MDN', 'DEBT_MALE_STGP_ANY_N', 'DEBT_MALE_STGP_ANY_MEAN', 'DEBT_MALE_STGP_ANY_MDN', 'DEBT_MALE_STGP_EVAL_N', 'DEBT_MALE_STGP_EVAL_MEAN', 'DEBT_MALE_STGP_EVAL_MDN', 'DEBT_MALE_PP_ANY_N', 'DEBT_MALE_PP_ANY_MEAN', 'DEBT_MALE_PP_ANY_MDN', 'DEBT_MALE_PP_EVAL_N', 'DEBT_MALE_PP_EVAL_MEAN', 'DEBT_MALE_PP_EVAL_MDN', 'DEBT_NOTMALE_STGP_ANY_N', 'DEBT_NOTMALE_STGP_ANY_MEAN', 'DEBT_NOTMALE_STGP_ANY_MDN', 'DEBT_NOTMALE_STGP_EVAL_N', 'DEBT_NOTMALE_STGP_EVAL_MEAN', 'DEBT_NOTMALE_STGP_EVAL_MDN', 'DEBT_NOTMALE_PP_ANY_N', 'DEBT_NOTMALE_PP_ANY_MEAN', 'DEBT_NOTMALE_PP_ANY_MDN', 'DEBT_NOTMALE_PP_EVAL_N', 'DEBT_NOTMALE_PP_EVAL_MEAN', 'DEBT_NOTMALE_PP_EVAL_MDN', 'DEBT_PELL_STGP_ANY_N', 'DEBT_PELL_STGP_ANY_MEAN', 'DEB

In [6]:
inst_path = RAW / "Most-Recent-Cohorts-Institution.csv"

header = pd.read_csv(inst_path, nrows=0)   # reads column names only
print(len(header.columns), "columns in the full file")

wanted = ["UNITID", "INSTNM", "STABBR", "CONTROL", "REGION", "LOCALE",
          "ADM_RATE", "UGDS", "PCTPELL", "C150_4"]
print("Missing:", [c for c in wanted if c not in header.columns])

3379 columns in the full file
Missing: []


In [7]:
inst = pd.read_csv(inst_path, usecols=wanted, dtype=str, low_memory=False)
print(inst.shape)

inst.to_csv(RAW / "institution_trimmed.csv", index=False)

(6278, 10)


In [8]:
from sqlalchemy import create_engine

engine = create_engine(
    "mssql+pyodbc://@localhost/Scorecard"
    "?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes",
    fast_executemany=True,
)

inst.to_sql("stg_institution", engine, if_exists="replace",
            index=False, chunksize=5000)
fos.to_sql("stg_field_of_study", engine, if_exists="replace",
           index=False, chunksize=5000)

-45